In [1]:
import pandas as pd
import numpy as np

print("⚙️ Gerando base de dados simulada para o projeto de Varejo...")

# Configuração de semente para manter os dados consistentes
np.random.seed(42)
n_registros = 1000

# 1. Gerando Dados Brutos de Vendas (Simulando o sistema OMS do E-commerce)
faturamento_fila = np.random.randint(100, 1500, size=n_registros)
regioes = np.random.choice([0, 1, 2, 3], size=n_registros, p=[0.4, 0.3, 0.2, 0.1]) # 0:SE, 1:S, 2:NE, 3:N
transp_id = np.random.choice([0, 1, 2], size=n_registros, p=[0.5, 0.3, 0.2])

dados_vendas = {
    'id_pedido_oms': [f'PED-{1000+i}' for i in range(n_registros)],
    'id_cliente': np.random.randint(10000, 99999, size=n_registros),
    'nome_cliente': [f'Cliente Academico {i}' for i in range(n_registros)],
    'cidade': np.random.choice(['São Paulo', 'Rio de Janeiro', 'Belo Horizonte', 'Porto Alegre', 'Recife', 'Salvador', 'Manaus'], size=n_registros),
    'estado': np.random.choice(['SP', 'RJ', 'MG', 'RS', 'PE', 'BA', 'AM'], size=n_registros),
    'regiao_destino': regioes,
    'id_transportadora': transp_id,
    'dt_compra': pd.date_range(start='2026-01-01', periods=n_registros, freq='h'),
    'vlr_frete_pago': np.round(np.random.uniform(15.00, 60.00, size=n_registros), 2),
    'qtd_itens': np.random.randint(1, 5, size=n_registros),
    'volume_fila_estoque': faturamento_fila,
    'dia_semana_faturamento': np.random.randint(0, 7, size=n_registros)
}
# Adicionando uma linha de teste para o algoritmo de limpeza remover
df_vendas_bruto = pd.DataFrame(dados_vendas)
df_vendas_bruto.loc[n_registros] = ['PED-TESTE', 99999, 'TESTE', 'SÃO PAULO', 'SP', 0, 0, pd.Timestamp('2026-01-01'), 0.00, 0, 0, 0]
df_vendas_bruto.to_csv('dados_vendas_brutos.csv', index=False)

# 2. Gerando Dados Brutos de Logística (Simulando arquivos de rastreamento TMS das Transportadoras)
# Criando uma lógica onde rotas para o Norte/Nordeste com a transportadora 2 geram atrasos reais (Lead time > 5 dias)
lead_time_base = np.random.randint(2, 5, size=n_registros)
atraso_induzido = (transp_id == 2) & (regioes >= 2)
lead_time_final = lead_time_base + np.where(atraso_induzido, np.random.randint(3, 7, size=n_registros), 0)

dt_entrega = df_vendas_bruto['dt_compra'].iloc[:-1] + pd.to_timedelta(lead_time_final, unit='D')

dados_logistica = {
    'cod_pedido_entrega': [f'PED-{1000+i}' for i in range(n_registros)],
    'razao_social': np.where(transp_id == 0, 'TransRápido Ltda', np.where(transp_id == 1, 'LogBrasil S/A', 'NorteFretes Express')),
    'dt_entrega_efetiva': dt_entrega,
    'vlr_custo_frete_real': np.round(df_vendas_bruto['vlr_frete_pago'].iloc[:-1] * np.random.uniform(0.85, 1.15, size=n_registros), 2)
}
df_logistica_bruto = pd.DataFrame(dados_logistica)

# Forçando alguns valores nulos e duplicatas para o script de ETL limpar
df_logistica_bruto.loc[10, 'dt_entrega_efetiva'] = None
df_logistica_bruto.loc[20, 'dt_entrega_efetiva'] = None
df_logistica_bruto.loc[n_registros] = ['PED-1005', 'TransRápido Ltda', pd.Timestamp('2026-01-02'), 25.00] # Duplicata

df_logistica_bruto.to_csv('dados_logistica_brutos.csv', index=False)
print("✅ Arquivos 'dados_vendas_brutos.csv' e 'dados_logistica_brutos.csv' criados com sucesso!")

⚙️ Gerando base de dados simulada para o projeto de Varejo...
✅ Arquivos 'dados_vendas_brutos.csv' e 'dados_logistica_brutos.csv' criados com sucesso!


In [2]:
# SCRIPT 1: PIPELINE DE ETL E MODELAGEM DIMENSIONAL (STAR SCHEMA)

import pandas as pd
import numpy as np

print("⚡ Iniciando a limpeza e processamento dos dados (ETL)...")

# 1. Carregando os arquivos brutos gerados
df_vendas = pd.read_csv('dados_vendas_brutos.csv')
df_logistica = pd.read_csv('dados_logistica_brutos.csv')

# 2. DATA QUALITY: Saneamento e Higienização de Dados
# A) Remoção de registros de testes do sistema
df_vendas = df_vendas[df_vendas['id_pedido_oms'] != 'PED-TESTE']

# B) Remoção de duplicidades no rastreio logístico (Mantendo a última atualização de entrega)
df_logistica = df_logistica.drop_duplicates(subset=['cod_pedido_entrega'], keep='last')

# C) Conversão dos campos de data de texto para Datetime
df_vendas['dt_compra'] = pd.to_datetime(df_vendas['dt_compra'])
df_logistica['dt_entrega_efetiva'] = pd.to_datetime(df_logistica['dt_entrega_efetiva'])

# D) Imputação de Valores Nulos (Pedidos sem data de entrega final recebem a média de trânsito de 4 dias)
df_logistica['dt_entrega_efetiva'] = df_logistica['dt_entrega_efetiva'].fillna(df_vendas['dt_compra'] + pd.Timedelta(days=4))

# 3. UNIÃO LÓGICA E REGRAS DE NEGÓCIO
df_provisorio = pd.merge(df_vendas, df_logistica, left_on='id_pedido_oms', right_on='cod_pedido_entrega', how='inner')

# Cálculo do Lead Time real em dias e checagem de estouro do SLA (Prazo máximo tolerado: 5 dias)
df_provisorio['lead_time_dias'] = (df_provisorio['dt_entrega_efetiva'] - df_provisorio['dt_compra']).dt.days
df_provisorio['flg_atraso'] = np.where(df_provisorio['lead_time_dias'] > 5, 1, 0)

# 4. EXTRAÇÃO DO MODELO DIMENSIONAL (STAR SCHEMA)
print("📐 Construindo tabelas fato e dimensões...")

# Criando a Dimensao_Cliente
dim_cliente = df_provisorio[['id_cliente', 'nome_cliente', 'cidade', 'estado', 'regiao_destino']].drop_duplicates().reset_index(drop=True)
dim_cliente.index.name = 'sk_cliente'
dim_cliente = dim_cliente.reset_index()

# Criando a Dimensao_Transportadora
dim_transportadora = df_provisorio[['id_transportadora', 'razao_social']].drop_duplicates().reset_index(drop=True)
dim_transportadora.index.name = 'sk_transportadora'
dim_transportadora = dim_transportadora.reset_index()

# Criando a Tabela Fato_Pedidos conectada pelas chaves substitutas (Surrogate Keys)
fato_pedidos = df_provisorio.merge(dim_cliente, on='id_cliente') \
                             .merge(dim_transportadora, on='id_transportadora')

fato_pedidos = fato_pedidos[[
    'sk_cliente', 'sk_transportadora', 'id_pedido_oms',
    'dt_compra', 'dt_entrega_efetiva', 'vlr_frete_pago',
    'vlr_custo_frete_real', 'qtd_itens', 'volume_fila_estoque',
    'dia_semana_faturamento', 'flg_atraso'
]]
fato_pedidos.index.name = 'sk_pedido'
fato_pedidos = fato_pedidos.reset_index()

# Exportando a estrutura limpa
dim_cliente.to_csv('dim_cliente.csv', index=False)
dim_transportadora.to_csv('dim_transportadora.csv', index=False)
fato_pedidos.to_csv('fato_pedidos.csv', index=False)

print("💾 Tabelas 'dim_cliente.csv', 'dim_transportadora.csv' e 'fato_pedidos.csv' geradas com sucesso!")

⚡ Iniciando a limpeza e processamento dos dados (ETL)...
📐 Construindo tabelas fato e dimensões...
💾 Tabelas 'dim_cliente.csv', 'dim_transportadora.csv' e 'fato_pedidos.csv' geradas com sucesso!


In [3]:
# SCRIPT 2: MOTOR DE ANALYTICS AVANÇADO (MACHINE LEARNING)

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("🤖 Iniciando o treinamento do modelo preditivo de Analytics...")

# 1. Lendo os dados higienizados da Tabela Fato
fato = pd.read_csv('fato_pedidos.csv')

# 2. Selecionando variáveis explicativas (Features) e a variável alvo (Target)
X = fato[['sk_transportadora', 'dia_semana_faturamento', 'volume_fila_estoque']]
y = fato['flg_atraso']

# Dividindo os dados: 70% para o modelo aprender (treino) e 30% para testar a precisão
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

# 3. Inicializando e treinando o classificador Random Forest
classificador_logistico = RandomForestClassifier(n_estimators=100, random_state=42)
classificador_logistico.fit(X_train, y_train)

# 4. Fazendo as previsões na base de teste
y_pred = classificador_logistico.predict(X_test)

# 5. GERANDO OS RESULTADOS EXIGIDOS PARA DOCUMENTAR NO RELATÓRIO
print("\n==================================================")
print("📊 INDICADORES DE PERFORMANCE DO MODELO (ANALYTICS)")
print("==================================================")
print(f"🎯 Acurácia do Modelo: {accuracy_score(y_test, y_pred) * 100:.2f}%")

print("\n🔍 Matriz de Confusão (Acertos vs Erros):")
print(confusion_matrix(y_test, y_pred))

print("\n📋 Relatório Estatístico Detalhado:")
print(classification_report(y_test, y_pred))

# Salvando a relevância dos fatores para justificar seus insights de negócio
importancias = classificador_logistico.feature_importances_
print("\n💡 Fatores que mais impactam no atraso das entregas:")
for coluna, peso in zip(X.columns, importancias):
    print(f" * {coluna}: {peso * 100:.2f}%")

🤖 Iniciando o treinamento do modelo preditivo de Analytics...

📊 INDICADORES DE PERFORMANCE DO MODELO (ANALYTICS)
🎯 Acurácia do Modelo: 96.46%

🔍 Matriz de Confusão (Acertos vs Erros):
[[381   7]
 [  7   1]]

📋 Relatório Estatístico Detalhado:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98       388
           1       0.12      0.12      0.12         8

    accuracy                           0.96       396
   macro avg       0.55      0.55      0.55       396
weighted avg       0.96      0.96      0.96       396


💡 Fatores que mais impactam no atraso das entregas:
 * sk_transportadora: 25.84%
 * dia_semana_faturamento: 8.39%
 * volume_fila_estoque: 65.77%
